In [ ]:
pip install langchain langchain-google-genai langchain-community langgraph

In [ ]:
import sqlite3

# Connect to database (creates students.db if it doesn't exist)
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

# Create table
cursor.execute(
    """
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
"""
)

# Insert student data
students_data = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88),
]

cursor.executemany(
    "INSERT OR REPLACE INTO students VALUES (?, ?, ?, ?, ?, ?, ?)", students_data
)
conn.commit()
conn.close()
print("Database created and populated successfully!")

Database created and populated successfully!


In [ ]:
import sqlite3
from langchain_core.tools import tool


# Tool 1: Get student info
@tool
def get_student_info(student_id: str) -> str:
    """Returns the student's name and department given their student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, department FROM students WHERE student_id = ?",
        (student_id,),
    )
    result = cursor.fetchone()
    conn.close()

    if result:
        return f"Name: {result[0]}, Department: {result[1]}"
    return "Student not found."


# Tool 2: Get student marks
@tool
def get_student_marks(student_id: str) -> str:
    """Returns individual subject marks (Python, Database, AI, Web) for a given student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT python, database, ai, web FROM students WHERE student_id = ?",
        (student_id,),
    )
    result = cursor.fetchone()
    conn.close()

    if result:
        return f"Python: {result[0]}, Database: {result[1]}, AI: {result[2]}, Web: {result[3]}"
    return "Student marks not found."


# Tool 3: Calculator
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression string to calculate totals or averages. Example input: '85 + 72 + 90 + 78' or '(85+72+90+78)/4'."""
    try:
        # Safe evaluation of mathematical expressions
        allowed_chars = "0123456789+-*/(). "
        if all(char in allowed_chars for char in expression):
            return str(eval(expression))
        return "Invalid math expression."
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"


# Updated Tool 4: Passing rules
@tool
def get_passing_rules(*args, **kwargs) -> str:
    """Returns the university passing criteria and rules.

    Does not require any arguments.
    """
    return "University Passing Rules: Minimum overall average required is 40%. Minimum mark required in each subject is 35%."

In [ ]:
import os
import sqlite3
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# 1. Set API Key
os.environ["GOOGLE_API_KEY"] = "API_KEY"

# 2. Define Tools
@tool
def get_student_info(student_id: str) -> str:
    """Returns the student's name and department given their student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, department FROM students WHERE student_id = ?",
        (student_id,),
    )
    result = cursor.fetchone()
    conn.close()
    return (
        f"Name: {result[0]}, Department: {result[1]}"
        if result
        else "Student not found."
    )

@tool
def get_student_marks(student_id: str) -> str:
    """Returns individual subject marks (Python, Database, AI, Web) for a given student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT python, database, ai, web FROM students WHERE student_id = ?",
        (student_id,),
    )
    result = cursor.fetchone()
    conn.close()
    return (
        f"Python: {result[0]}, Database: {result[1]}, AI: {result[2]}, Web: {result[3]}"
        if result
        else "Student marks not found."
    )

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression string to calculate totals or averages. Example: '85 + 72 + 90 + 78'."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(char in allowed_chars for char in expression):
            return str(eval(expression))
        return "Invalid math expression."
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_passing_rules(*args, **kwargs) -> str:
    """Returns the university passing criteria and rules. Takes no required parameters."""
    return "University Passing Rules: Minimum overall average required is 40%. Minimum mark required in each subject is 35%."

# 3. Initialize Gemini with a valid model identifier
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# 4. Create Agent using standard LangGraph prebuilt creator
tools = [get_student_info, get_student_marks, calculator, get_passing_rules]
agent_executor = create_react_agent(llm, tools)

# 5. Run Query
query = "I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements."
response = agent_executor.invoke({"messages": [HumanMessage(content=query)]})

# Print Answer
print("\n--- FINAL ANSWER ---")
print(response["messages"][-1].content)

/tmp/ipykernel_6002/407486380.py:67: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.



--- FINAL ANSWER ---
[{'type': 'text', 'text': 'Here are your details for Student ID **22CS045**:\n\n* **Name:** Dhanushya\n* **Department:** Computer Science\n\n### **Marks Overview:**\n* **Python:** 85\n* **Database:** 72\n* **AI:** 90\n* **Web:** 78\n* **Total Marks:** 325 / 400\n* **Average Marks:** 81.25%\n\n### **Passing Status:**\n* **University Rules:** Minimum 35% in each subject and an overall average of at least 40%.\n* **Result:** **PASSED** (You satisfy all university passing requirements).', 'extras': {'signature': 'EpQGCpEGAWkUfRMFRDHCIOIGQPhh2xjDEv0U61vg5bm1sfnMjn91zGQ2CmNfDy6qmhSB2bGVGf8je3c+C/urVVGW7j11heNq8sWZrI5b0z1C2fm5tK9AMAHnXG+RICf+rFl0g995tMmLf7NWaTHiP+n4Dma5rIOGATo3aEyldXgCe72mk6KYUACO2yQN4z1Ux1nRf4fNGWzdiOnyPzjGhlgECnDxE7abK+IyGZVGUNQ27GI9N12g8cP+SCpTbFqlyrdVyfgVSnalbRVbwcL0+um86vuo3XSlTRvtjqpG+g9Z+3YfY5amJzK3b7MTA1jHPWHoBT4Jgqu9tkXRawTzpznzRHXaqPMxlH+kc6tWCpUR78aH1lSyv6iNVy/nSqOJzO52M6rmxbFlCh8FCxj2y2S2WHtXQtgi74hLCXTlf23vOK40MLPImRxH3EtmHz+QVSgDqZqWGw/IWHA

In [ ]:
# Print the final answer stored in the response to verify the output
print("--- FINAL ANSWER FROM RESPONSE ---")
print(response["messages"][-1].content)

--- FINAL ANSWER FROM RESPONSE ---
[{'type': 'text', 'text': 'Here are your details for Student ID **22CS045**:\n\n* **Name:** Dhanushya\n* **Department:** Computer Science\n\n### **Marks Overview:**\n* **Python:** 85\n* **Database:** 72\n* **AI:** 90\n* **Web:** 78\n* **Total Marks:** 325 / 400\n* **Average Marks:** 81.25%\n\n### **Passing Status:**\n* **University Rules:** Minimum 35% in each subject and an overall average of at least 40%.\n* **Result:** **PASSED** (You satisfy all university passing requirements).', 'extras': {'signature': 'EpQGCpEGAWkUfRMFRDHCIOIGQPhh2xjDEv0U61vg5bm1sfnMjn91zGQ2CmNfDy6qmhSB2bGVGf8je3c+C/urVVGW7j11heNq8sWZrI5b0z1C2fm5tK9AMAHnXG+RICf+rFl0g995tMmLf7NWaTHiP+n4Dma5rIOGATo3aEyldXgCe72mk6KYUACO2yQN4z1Ux1nRf4fNGWzdiOnyPzjGhlgECnDxE7abK+IyGZVGUNQ27GI9N12g8cP+SCpTbFqlyrdVyfgVSnalbRVbwcL0+um86vuo3XSlTRvtjqpG+g9Z+3YfY5amJzK3b7MTA1jHPWHoBT4Jgqu9tkXRawTzpznzRHXaqPMxlH+kc6tWCpUR78aH1lSyv6iNVy/nSqOJzO52M6rmxbFlCh8FCxj2y2S2WHtXQtgi74hLCXTlf23vOK40MLPImRxH3EtmHz+QVS